In [ ]:
!pip install datasets transformers manga109api Pillow torch torchvision ultralytics

from google.colab import userdata
from datasets import load_dataset
import zipfile
import manga109api
import os

from ultralytics import YOLO
from PIL import Image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="hal-utokyo/Manga109",
    repo_type="dataset",
    local_dir="/content/manga109"
)
print(path)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/content/manga109


In [ ]:
for root, dirs, files in os.walk("/content/manga109"):
    level = root.replace("/content/manga109", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # only show files for top 2 levels
        for f in files[:5]:  # limit to 5 files per folder
            print(f"{indent}  {f}")

manga109/
  Manga109_released_2023_12_07.zip
  README.md
  .gitattributes
  .cache/
    huggingface/
      download/


In [ ]:
zip_path = "/content/manga109/Manga109_released_2023_12_07.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist()[:20])  # preview contents first

['Manga109_released_2023_12_07/', '__MACOSX/._Manga109_released_2023_12_07', 'Manga109_released_2023_12_07/books.txt', '__MACOSX/Manga109_released_2023_12_07/._books.txt', 'Manga109_released_2023_12_07/annotations.v2020.12.18/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2020.12.18', 'Manga109_released_2023_12_07/images/', '__MACOSX/Manga109_released_2023_12_07/._images', 'Manga109_released_2023_12_07/annotations.v2018.05.31/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2018.05.31', 'Manga109_released_2023_12_07/annotations/', '__MACOSX/Manga109_released_2023_12_07/._annotations', 'Manga109_released_2023_12_07/readme.txt', '__MACOSX/Manga109_released_2023_12_07/._readme.txt', 'Manga109_released_2023_12_07/annotations_COO/', '__MACOSX/Manga109_released_2023_12_07/._annotations_COO', 'Manga109_released_2023_12_07/annotations_Manga109Dialog/', '__MACOSX/Manga109_released_2023_12_07/._annotations_Manga109Dialog', 'Manga109_released_2023_12_07/annotations.v2020.12.18/M

In [ ]:
extract_path = "/content/manga109_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    members = [m for m in z.namelist() if not m.startswith("__MACOSX")]
    z.extractall(extract_path, members=members)

print("Done!")

Done!


In [ ]:
data_root = "/content/manga109_data/Manga109_released_2023_12_07"
api = manga109api.Parser(root_dir=data_root)

# List all books
print(api.books)

['ARMS', 'AisazuNihaIrarenai', 'AkkeraKanjinchou', 'Akuhamu', 'AosugiruHaru', 'AppareKappore', 'Arisa', 'BEMADER_P', 'BakuretsuKungFuGirl', 'Belmondo', 'BokuHaSitatakaKun', 'BurariTessenTorimonocho', 'ByebyeC-BOY', 'Count3DeKimeteAgeru', 'DollGun', 'Donburakokko', 'DualJustice', 'EienNoWith', 'EvaLady', 'EverydayOsakanaChan', 'GOOD_KISS_Ver2', 'GakuenNoise', 'GarakutayaManta', 'GinNoChimera', 'Hamlet', 'HanzaiKousyouninMinegishiEitarou', 'HaruichibanNoFukukoro', 'HarukaRefrain', 'HealingPlanet', 'HeiseiJimen', 'HighschoolKimengumi_vol01', 'HighschoolKimengumi_vol20', 'HinagikuKenzan', 'HisokaReturns', 'JangiriPonpon', 'JijiBabaFight', 'Joouari', 'Jyovolley', 'KarappoHighschool', 'KimiHaBokuNoTaiyouDa', 'KoukouNoHitotachi', 'KuroidoGanka', 'KyokugenCyclone', 'LancelotFullThrottle', 'LoveHina_vol01', 'LoveHina_vol14', 'MAD_STONE', 'MadouTaiga', 'MagicStarGakuin', 'MagicianLoad', 'MariaSamaNihaNaisyo', 'MayaNoAkaiKutsu', 'MemorySeijin', 'MeteoSanStrikeDesu', 'MiraiSan', 'MisutenaideDaisy'

In [ ]:
import os
from PIL import Image

LABEL2ID = {"body": 0, "text": 1}

def convert_all_annotations(api, data_root, label_out_root):
    image_dir = os.path.join(data_root, "images")

    for book in api.books:
        ann = api.get_annotation(book=book)

        for page in ann['page']:
            page_index = page['@index']
            W = page['@width']
            H = page['@height']

            img_filename = f"{page_index:03d}.jpg"
            img_path = os.path.join(image_dir, book, img_filename)
            if not os.path.exists(img_path):
                continue

            lines = []
            for tag, label_id in LABEL2ID.items():
                for item in page[tag]:
                    x1 = item['@xmin']
                    y1 = item['@ymin']
                    x2 = item['@xmax']
                    y2 = item['@ymax']

                    cx = ((x1 + x2) / 2) / W
                    cy = ((y1 + y2) / 2) / H
                    w  = (x2 - x1) / W
                    h  = (y2 - y1) / H
                    lines.append(f"{label_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            label_out = os.path.join(label_out_root, book)
            os.makedirs(label_out, exist_ok=True)
            with open(os.path.join(label_out, img_filename.replace(".jpg", ".txt")), "w") as f:
                f.write("\n".join(lines))

    print("Annotation conversion done!")

label_out_root = "/content/manga109_labels"
convert_all_annotations(api, data_root, label_out_root)

Annotation conversion done!


In [ ]:
# Verify a label file looks right
import random
book = random.choice(api.books)
label_file = f"/content/manga109_labels/{book}/002.txt"
if os.path.exists(label_file):
    with open(label_file) as f:
        print(f.read())

0 0.751511 0.456838 0.469166 0.679487
0 0.249698 0.499573 0.469166 0.999145
1 0.894800 0.175214 0.060459 0.206838
1 0.591294 0.205983 0.065296 0.208547
1 0.896312 0.584188 0.045345 0.204274
1 0.720677 0.863248 0.065296 0.177778


In [ ]:
import shutil, yaml, os

data_root = "/content/manga109_data/Manga109_released_2023_12_07"
image_dir = os.path.join(data_root, "images")
label_out_root = "/content/manga109_labels"
dataset_root = "/content/manga109_dataset"

# Train/val split (80/20)
all_books = api.books
split = int(len(all_books) * 0.8)
train_books = all_books[:split]
val_books = all_books[split:]

print(f"Train: {len(train_books)} books, Val: {len(val_books)} books")

# Copy images and labels into dataset folder
for split_name, books in [("train", train_books), ("val", val_books)]:
    img_split_dir = os.path.join(dataset_root, "images", split_name)
    lbl_split_dir = os.path.join(dataset_root, "labels", split_name)
    os.makedirs(img_split_dir, exist_ok=True)
    os.makedirs(lbl_split_dir, exist_ok=True)

    for book in books:
        src_img = os.path.join(image_dir, book)
        src_lbl = os.path.join(label_out_root, book)
        dst_img = os.path.join(img_split_dir, book)
        dst_lbl = os.path.join(lbl_split_dir, book)

        if os.path.exists(src_img) and not os.path.exists(dst_img):
            shutil.copytree(src_img, dst_img)
        if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
            shutil.copytree(src_lbl, dst_lbl)

print("Dataset split done!")

Train: 87 books, Val: 22 books
Dataset split done!


In [ ]:

import os, sys, shutil, yaml, json, time
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import manga109api


data_root    = "/content/manga109_data/Manga109_released_2023_12_07"
dataset_root = "/content/manga109_dataset"
api          = manga109api.Parser(root_dir=data_root)

dataset_cfg = {
    "path":  dataset_root,
    "train": "images/train",
    "val":   "images/val",
    "nc":    2,
    "names": {0: "body", 1: "text"},
}
yaml_path = os.path.join(dataset_root, "manga109.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(dataset_cfg, f)
print(f"Dataset YAML written → {yaml_path} ✓")

# These modules were retrieved from online
class ConvIN(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, s, k // 2, bias=False)
        self.norm = nn.InstanceNorm2d(out_ch, affine=True)
        self.act  = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class ViTBlock(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        while dim % num_heads != 0 and num_heads > 1:
            num_heads -= 1
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        t = x.flatten(2).permute(0, 2, 1)
        t = t + self.attn(*([self.norm1(t)] * 3))[0]
        t = t + self.ffn(self.norm2(t))
        return t.permute(0, 2, 1).reshape(B, C, H, W)


class SPPFWithViT(nn.Module):
    def __init__(self, sppf, vit):
        super().__init__()
        self.sppf = sppf
        self.vit  = vit
        self.i    = sppf.i
        self.f    = sppf.f
        self.type = "SPPFWithViT"

    def forward(self, x):
        return self.vit(self.sppf(x))


# AI DISCLAIMER: AI WAS USED TO USE PATCHTRAINER TO SUB IN MODULE. WE WERE UNABLE TO FIGURE OUT HOW TO DO IT OURSELVES.
class PatchedTrainer(DetectionTrainer):
    """
    Overrides get_model() so ultralytics never rebuilds the
    architecture from YAML. Returns our patched model directly.
    """
    def __init__(self, patched_model, overrides=None):
        self._patched_model = patched_model
        super().__init__(overrides=overrides)

    def get_model(self, cfg=None, weights=None, verbose=True):
        self._patched_model.nc = self.data['nc']
        return self._patched_model


base     = YOLO("yolov8s.pt")
detector = base.model

# Patch layers 0 and 1 → ConvIN
for idx, (in_ch, out_ch) in enumerate([(3, 32), (32, 64)]):
    old = detector.model[idx]
    new = ConvIN(in_ch, out_ch, k=3, s=2)
    new.i = old.i
    new.f = old.f
    new.type = "ConvIN"
    detector.model[idx] = new
print("Layers 0,1 → ConvIN ✓")

# Patch layer 9 → SPPF + ViTBlock
old_sppf = detector.model[9]
detector.model[9] = SPPFWithViT(old_sppf, ViTBlock(dim=512, num_heads=8))
print("Layer 9 → SPPF+ViT ✓")

# Verify forward pass
detector.cuda()
dummy = torch.randn(1, 3, 640, 640).cuda()
with torch.no_grad():
    detector(dummy)
print("Forward pass OK ✓")

# Print layer table
print("\nLayer verification:")
for i, layer in enumerate(detector.model):
    marker = " ← CUSTOM" if type(layer).__name__ in ('ConvIN', 'SPPFWithViT') else ""
    print(f"  Layer {i:2d} | {type(layer).__name__:20s}{marker}")

# ── Train via PatchedTrainer ───────────────────────────────────────
args = dict(
    data         = yaml_path,
    epochs       = 20,
    imgsz        = 640,
    batch        = 8,
    name         = "manga109_v1_shape_backbone",
    project      = "/content/runs",
    device       = 0,
    optimizer    = "AdamW",
    lr0          = 5e-4,
    warmup_epochs= 3,
    hsv_s        = 0.5,
    mixup        = 0.1,
    model        = "yolov8s.pt",  # needed for cfg parsing, ignored in get_model
)

trainer = PatchedTrainer(patched_model=detector, overrides=args)
trainer.train()
print("Training complete ✓")



Dataset YAML written → /content/manga109_dataset/manga109.yaml ✓
Layers 0,1 → ConvIN ✓
Layer 9 → SPPF+ViT ✓
Forward pass OK ✓

Layer verification:
  Layer  0 | ConvIN               ← CUSTOM
  Layer  1 | ConvIN               ← CUSTOM
  Layer  2 | C2f                 
  Layer  3 | Conv                
  Layer  4 | C2f                 
  Layer  5 | Conv                
  Layer  6 | C2f                 
  Layer  7 | Conv                
  Layer  8 | C2f                 
  Layer  9 | SPPFWithViT          ← CUSTOM
  Layer 10 | Upsample            
  Layer 11 | Concat              
  Layer 12 | C2f                 
  Layer 13 | Upsample            
  Layer 14 | Concat              
  Layer 15 | C2f                 
  Layer 16 | Conv                
  Layer 17 | Concat              
  Layer 18 | C2f                 
  Layer 19 | Conv                
  Layer 20 | Concat              
  Layer 21 | C2f                 
  Layer 22 | Detect              
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL ultralytics.nn.tasks.DetectionModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([ultralytics.nn.tasks.DetectionModel])` or the `torch.serialization.safe_globals([ultralytics.nn.tasks.DetectionModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of to

In [ ]:
# ── Evaluate ───────────────────────────────────────────────────────
weights_path = os.path.join(trainer.save_dir, "weights", "best.pt")
print(f"\nLoading best weights from {weights_path}")

# Reload patched model with best weights for eval
ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
detector_eval = ckpt.get('ema') or ckpt['model']
detector_eval = detector_eval.float().cuda()

# Wrap in YOLO for .val()
eval_yolo       = YOLO("yolov8s.pt")
eval_yolo.model = detector_eval
metrics         = eval_yolo.val(data=yaml_path, imgsz=640, batch=8)

map50             = metrics.box.map50
map5095           = metrics.box.map
map50_per_class   = metrics.box.ap50
map5095_per_class = metrics.box.ap
precision         = metrics.box.mp
recall            = metrics.box.mr
f1                = 2 * (precision * recall) / (precision + recall + 1e-8)

print(f"\nmAP50:     {map50:.4f}")
print(f"mAP50-95:  {map5095:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

# ── Inference speed ───────────────────────────────────────────────
sample_img = os.path.join(data_root, "images", api.books[0], "002.jpg")
for _ in range(3):
    eval_yolo(sample_img, verbose=False)

times = []
for _ in range(50):
    start = time.perf_counter()
    eval_yolo(sample_img, verbose=False)
    times.append((time.perf_counter() - start) * 1000)

avg_ms = np.mean(times)
fps    = 1000 / avg_ms
print(f"Latency: {avg_ms:.2f} ms | FPS: {fps:.1f}")

# ── Save results ──────────────────────────────────────────────────
model_size_mb = os.path.getsize(weights_path) / (1024 ** 2)
param_count   = sum(p.numel() for p in detector_eval.parameters())

results = {
    "model":         "YOLOv8s_V1_ShapeBackbone",
    "mAP50":         round(map50, 4),
    "mAP50-95":      round(map5095, 4),
    "precision":     round(precision, 4),
    "recall":        round(recall, 4),
    "f1":            round(f1, 4),
    "AP50_body":     round(map50_per_class[0], 4),
    "AP50_text":     round(map50_per_class[1], 4),
    "AP5095_body":   round(map5095_per_class[0], 4),
    "AP5095_text":   round(map5095_per_class[1], 4),
    "latency_ms":    round(avg_ms, 2),
    "fps":           round(fps, 1),
    "model_size_mb": round(model_size_mb, 2),
    "params_M":      round(param_count / 1e6, 2),
}

print("\n── Summary ──")
for k, v in results.items():
    print(f"  {k}: {v}")

with open("/content/results_v1_shape_backbone.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved → /content/results_v1_shape_backbone.json")

# ── Save to Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/drive')

shutil.copytree(
    str(trainer.save_dir),
    "/drive/MyDrive/manga109_v1_shape_backbone",
    dirs_exist_ok=True
)
shutil.copy(
    "/content/results_v1_shape_backbone.json",
    "/drive/MyDrive/results_v1_shape_backbone.json"
)
print("Saved to Drive ✓")


Loading best weights from /content/runs/manga109_v1_shape_backbone-4/weights/best.pt
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 84 layers, 14,309,024 parameters, 0 gradients, 30.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2419.0±93.8 MB/s, size: 373.1 KB)
val: Scanning /content/manga109_dataset/labels/val/TapkunNoTanteisitsu.cache... 2077 images, 94 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2077/2077 414.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 260/260 6.2it/s 41.7s
                   all       2077      59398      0.906      0.851      0.922      0.672
                  body       1979      31178      0.892      0.797      0.898       0.64
                  text       1962      28220       0.92      0.904      0.946      0.704
Speed: 1.3ms preprocess, 8.1ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /co